# hudukaata — Indexer (Google Colab)

Indexes your Google Drive media (photos, videos, audio) into a semantic vector
database stored back on Google Drive.

**Run all cells in order, top to bottom.** Re-running individual cells out of
order may cause import errors.

**Steps:**
1. Configure your Drive paths and options (Cell 1)
2. Install packages (Cell 2)
3. Mount Google Drive (Cell 3)
4. Run the indexer (Cell 4)

Start with a small folder of test photos to verify everything works before
indexing your full library.

## 1 — Configuration

Edit the variables below, then run the remaining cells in order.

| Variable | Description |
|---|---|
| `MEDIA_DRIVE_PATH` | Subfolder inside MyDrive where your media files live. Use `""` for the Drive root. |
| `STORE_DRIVE_PATH` | Subfolder inside MyDrive where the index will be written. Created if absent. |
| `CAPTION_MODEL` | `"blip2_faces"` — captions + face detection (recommended). `"blip2"` — captions only, faster but no face search. |
| `FOLDER` | Subfolder inside `MEDIA_DRIVE_PATH` to index. `None` scans everything. |
| `CHECKPOINT_INTERVAL` | Save a recovery checkpoint every N media items. `0` = after every item; `-1` = disabled. |
| `CLUSTER_THRESHOLD` | Face clustering sensitivity (0–1). Lower = more clusters. Only used when `CAPTION_MODEL="blip2_faces"`. |

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
MEDIA_DRIVE_PATH    = "Photos"           # subfolder in MyDrive; "" for Drive root
STORE_DRIVE_PATH    = "hudukaata-store"  # will be created if it does not exist
CAPTION_MODEL       = "blip2_faces"      # "blip2_faces" | "blip2"
FOLDER              = None               # e.g. "2024/vacations" or None
CHECKPOINT_INTERVAL = 100                # number of media items between checkpoint saves
CLUSTER_THRESHOLD   = 0.6               # face clustering threshold (blip2_faces only)
# IO prefetch — overlap Drive reads with GPU inference (recommended for Colab)
PREFETCH         = True                 # set False to disable
MAX_PREFETCH     = 8                    # max pre-loaded images in memory (~36 MB each)
PREFETCH_WORKERS = 4                    # parallel Drive read threads
# ──────────────────────────────────────────────────────────────────────────────

assert CAPTION_MODEL in ("blip2", "blip2_faces"), (
    f"Unknown CAPTION_MODEL {CAPTION_MODEL!r}. Must be 'blip2' or 'blip2_faces'."
)

## 2 — Install packages

Installs `common` and `indexer` from the hudukaata Git repository.
Takes a few minutes on first run; subsequent runs reuse the Colab cache.

In [ ]:
BRANCH = "main"
# BRANCH = "claude/add-colab-run-feature-KFPa5"  # use this branch before it is merged

import subprocess, sys

def _pip(*args):
    # -q suppresses progress bars but keeps warnings visible.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip(f"git+https://github.com/kusimari/hudukaata@{BRANCH}#subdirectory=common")
_pip(f"git+https://github.com/kusimari/hudukaata@{BRANCH}#subdirectory=indexer")

## 3 — Mount Google Drive

A browser pop-up will ask you to authorise access. Drive is mounted at
`/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive as _colab_drive
_colab_drive.mount("/content/drive", force_remount=False)
print("Drive mounted at /content/drive/MyDrive/")

## 4 — Run the indexer

Builds the indexer pipeline from the project classes directly and runs it.
Progress is logged to the cell output. Interrupted runs can be resumed — the
indexer skips files whose modification time has not changed since the last run.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

from common.base import StorePointer
from common.media import GdriveMediaSource
from indexer.batch import AdaptiveBatchController
from indexer.pipeline import AdaptiveBatchRunner
from indexer.runner import IndexingRunner

media = GdriveMediaSource(MEDIA_DRIVE_PATH)
store = StorePointer.parse(f"file:///content/drive/MyDrive/{STORE_DRIVE_PATH}")

ctrl   = AdaptiveBatchController(initial_size=1, max_size=8, adaptive=True)
runner = IndexingRunner(
    pipeline_runner=AdaptiveBatchRunner(ctrl),
    checkpoint_interval=CHECKPOINT_INTERVAL,
    prefetch=PREFETCH,
    max_prefetch=MAX_PREFETCH,
    prefetch_workers=PREFETCH_WORKERS,
)

if CAPTION_MODEL == "blip2_faces":
    from indexer.indexers.blip2_sentok_exif_insightface_chroma import (
        Blip2SentTokExifInsightfaceChromaIndexer,
    )
    from indexer.models.blip2 import Blip2CaptionModel
    from indexer.models.insightface import InsightFaceModel
    from indexer.stores.chroma_caption import ChromaCaptionIndexStore
    from indexer.stores.chroma_face import ChromaFaceIndexStore

    caption_store = ChromaCaptionIndexStore()
    face_store    = ChromaFaceIndexStore()
    indexer = Blip2SentTokExifInsightfaceChromaIndexer(
        caption_model=Blip2CaptionModel(load_in_8bit=False),
        face_model=InsightFaceModel(),
        caption_store=caption_store,
        face_store=face_store,
        cluster_threshold=CLUSTER_THRESHOLD,
    )
    runner.run(
        pipeline=indexer.pipeline(),
        media=media,
        store=store,
        index_store=caption_store,
        index_store_name="indexer.stores.chroma_caption.ChromaCaptionIndexStore",
        folder=FOLDER,
        secondary_stores=[
            (face_store, "indexer.stores.chroma_face.ChromaFaceIndexStore"),
        ],
    )

else:  # blip2
    from indexer.indexers.blip2_sentok_exif_chroma import Blip2SentTokExifChromaIndexer
    from indexer.models.blip2 import Blip2CaptionModel
    from indexer.stores.chroma_caption import ChromaCaptionIndexStore

    caption_store = ChromaCaptionIndexStore()
    indexer = Blip2SentTokExifChromaIndexer(
        caption_model=Blip2CaptionModel(load_in_8bit=False),
        index_store=caption_store,
    )
    runner.run(
        pipeline=indexer.pipeline(),
        media=media,
        store=store,
        index_store=caption_store,
        index_store_name="indexer.stores.chroma_caption.ChromaCaptionIndexStore",
        folder=FOLDER,
    )

print(f"\nIndexing complete. Index written to /content/drive/MyDrive/{STORE_DRIVE_PATH}")